In [1]:
pip install pandas sqlalchemy openpyxl

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   -- ----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import re
import pandas as pd
from sqlalchemy import create_engine

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
FOLDER = "E-Mobility datasets"                 # main folder
DB_URL = "sqlite:///emobility.db"              # or mysql/postgres/sqlserver
IF_EXISTS = "replace"                          # "replace" or "append"
SKIP_HIDDEN = True                             # skip files starting with "." or "~$"

# ─────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────
def clean_name(name: str) -> str:
    """Turn any string into a safe SQL identifier."""
    name = re.sub(r"\W+", "_", str(name))      # non-word chars → _
    name = re.sub(r"_+", "_", name)            # collapse multiple _
    name = name.strip("_").lower()
    if name and name[0].isdigit():             # can't start with digit
        name = "t_" + name
    return name or "table"


def robust_read_csv(path: str) -> pd.DataFrame:
    """Try multiple strategies until the CSV loads."""
    attempts = [
        {"encoding": "utf-8"},
        {"encoding": "utf-8-sig"},
        {"encoding": "latin-1"},
        {"encoding": "cp1252"},
        {"encoding": "utf-8", "sep": ";"},
        {"encoding": "latin-1", "sep": ";"},
        {"encoding": "utf-8", "sep": "\t"},
        {"encoding": "latin-1", "sep": "\t"},
        {"encoding": "utf-8", "sep": None, "engine": "python"},
        {"encoding": "latin-1", "sep": None, "engine": "python"},
        {"encoding": "utf-8", "on_bad_lines": "skip"},
        {"encoding": "latin-1", "on_bad_lines": "skip"},
        {"encoding": "utf-8", "sep": None, "engine": "python", "on_bad_lines": "skip"},
        {"encoding": "latin-1", "sep": None, "engine": "python", "on_bad_lines": "skip"},
    ]
    last_err = None
    for kwargs in attempts:
        try:
            return pd.read_csv(path, **kwargs)
        except Exception as e:
            last_err = e
    raise last_err


def load_file(path: str) -> pd.DataFrame | None:
    """Dispatch to the right pandas reader based on extension."""
    lower = path.lower()
    if lower.endswith(".csv"):
        return robust_read_csv(path)
    if lower.endswith((".xlsx", ".xls")):
        return pd.read_excel(path)
    if lower.endswith(".json"):
        return pd.read_json(path)
    if lower.endswith(".parquet"):
        return pd.read_parquet(path)
    if lower.endswith(".txt"):
        # try tab first, then comma
        try:
            return pd.read_csv(path, sep="\t", encoding="latin-1")
        except Exception:
            return pd.read_csv(path, encoding="latin-1")
    return None


def make_table_name(root: str, filename: str, base_folder: str) -> str:
    """Build a table name, prefixing subfolder path to avoid collisions."""
    rel_dir = os.path.relpath(root, base_folder)
    base = os.path.splitext(filename)[0]
    if rel_dir in (".", ""):
        return clean_name(base)
    return clean_name(f"{rel_dir}_{base}")


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    if not os.path.isdir(FOLDER):
        print(f"❌ Folder not found: {FOLDER}")
        return

    engine = create_engine(DB_URL)

    total_ok = 0
    total_skip = 0
    total_fail = 0
    seen_tables = {}

    for root, dirs, files in os.walk(FOLDER):
        for filename in sorted(files):
            # skip hidden / temp files
            if SKIP_HIDDEN and (filename.startswith(".") or filename.startswith("~$")):
                total_skip += 1
                continue

            path = os.path.join(root, filename)

            try:
                df = load_file(path)
                if df is None:
                    print(f"⏭️  Skipped (unsupported): {path}")
                    total_skip += 1
                    continue

                if df.empty:
                    print(f"⏭️  Skipped (empty): {path}")
                    total_skip += 1
                    continue

                # clean column names
                df.columns = [clean_name(c) for c in df.columns]

                # drop completely empty rows/columns
                df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

                # table name (with subfolder prefix)
                table = make_table_name(root, filename, FOLDER)

                # handle duplicates: append a counter if the table name repeats
                if table in seen_tables:
                    seen_tables[table] += 1
                    table = f"{table}_{seen_tables[table]}"
                else:
                    seen_tables[table] = 1

                df.to_sql(table, engine, if_exists=IF_EXISTS, index=False)
                print(f"✅ {os.path.relpath(path, FOLDER)}  →  table '{table}'  "
                      f"({len(df)} rows, {len(df.columns)} cols)")
                total_ok += 1

            except Exception as e:
                print(f"❌ Failed: {os.path.relpath(path, FOLDER)}")
                print(f"   {type(e).__name__}: {e}")
                total_fail += 1

    print("\n" + "─" * 60)
    print(f"Done.  ✅ {total_ok} loaded   ⏭️  {total_skip} skipped   ❌ {total_fail} failed")


# ─────────────────────────────────────────────────────────────
# VERIFY
# ─────────────────────────────────────────────────────────────
def show_tables():
    from sqlalchemy import inspect
    engine = create_engine(DB_URL)
    insp = inspect(engine)
    tables = insp.get_table_names()
    print(f"\n📚 Tables in database ({len(tables)}):")
    for t in tables:
        print(f"   - {t}")


if __name__ == "__main__":
    main()
    show_tables()

✅ E_Mobility_Data_Template_SOW_4.1.xlsx  →  table 'e_mobility_data_template_sow_4_1'  (5 rows, 1 cols)
✅ FOR PUBLICATION_Africa EV Readiness_impact Indicator Database 2025.xlsx  →  table 'for_publication_africa_ev_readiness_impact_indicator_database_2025'  (54 rows, 1 cols)
✅ UNEP_EAC_EMobility_Gantt_Chart.xlsx  →  table 'unep_eac_emobility_gantt_chart'  (12 rows, 31 cols)
✅ UNEP_EAC_EMobility_Weekly_Live_Chart.xlsx  →  table 'unep_eac_emobility_weekly_live_chart'  (26 rows, 6 cols)
✅ battery_eol.csv  →  table 'battery_eol'  (5 rows, 9 cols)
✅ country_qualitative.csv  →  table 'country_qualitative'  (5 rows, 9 cols)
✅ eac_countries.csv  →  table 'eac_countries'  (8 rows, 8 cols)
✅ financing.csv  →  table 'financing'  (7 rows, 9 cols)
✅ fiscal_tariffs.csv  →  table 'fiscal_tariffs'  (8 rows, 9 cols)
✅ industrial_value_chain.csv  →  table 'industrial_value_chain'  (11 rows, 10 cols)
✅ kenya_ev_charging_infrastructure.csv  →  table 'kenya_ev_charging_infrastructure'  (29 rows, 14 cols)
✅ 

In [4]:
import sqlite3

conn = sqlite3.connect("emobility.db")

In [5]:
#getting database names
cursor = conn.cursor()
cursor.execute("""SELECT name
FROM sqlite_master
WHERE type = 'table';""")
print(cursor.fetchall())

[('e_mobility_data_template_sow_4.1',), ('kenya_ev_stations_(1)',), ('e_mobility_data_template_sow_4_1',), ('for_publication_africa_ev_readiness_impact_indicator_database_2025',), ('unep_eac_emobility_gantt_chart',), ('unep_eac_emobility_weekly_live_chart',), ('battery_eol',), ('country_qualitative',), ('eac_countries',), ('financing',), ('fiscal_tariffs',), ('industrial_value_chain',), ('kenya_ev_charging_infrastructure',), ('kenya_ev_stations_1',), ('market_fleet',), ('policy_inventory',), ('power_systems',), ('rwanda_ev_charging_stations_1',), ('tanzania_ev_charging_stations',), ('transition_control_fuel_motorcycle_daily_data',), ('transition_control_fuel_motorcycle_trip_data',), ('transition_treatment_electric_motorcycle_daily_data',), ('transition_treatment_electric_motorcycle_trip_data',), ('uganda_ev_charging_stations_dataset',)]


In [6]:
import pandas as pd 
pd.read_sql("""SELECT name FROM sqlite_master WHERE type='table';""", conn)

,name
0,e_mobility_data_template_sow_4.1
1,kenya_ev_stations_(1)
2,e_mobility_data_template_sow_4_1
3,for_publication_africa_ev_readiness_impact_ind...
4,unep_eac_emobility_gantt_chart
5,unep_eac_emobility_weekly_live_chart
6,battery_eol
7,country_qualitative
8,eac_countries
9,financing
